sun mapping stuff by tachyon

In [4]:
try:
    import fermipy
    print("Ready.")
except ImportError:
    print("Fermipy is not installed. Please install it to use this script.")

Ready.


imported stuff/setup

In [2]:
import pandas as pd
from astropy.time import Time
from astropy.coordinates import get_sun
import astropy.units as u
from fermipy.gtanalysis import GTAnalysis
import os
from pathlib import Path
from astropy.io import fits 


i want to work on how to get the correct solar flares myself from the fermi lat data query

In [12]:

#read the original csv file from the extra stuff folder 
df = pd.read_csv("extrastuff/LocalizationTable.csv")

#buffer for time before and after events
buffer = 10

print("Fermi LAT Data Query Parameters:")

#get the information from the csv and convert to gregorian
for index, row in df.iterrows():
    flarename = str(row["Indv Name"])
    date_time_i = str(row["Date and Time"])

    date_time_f = date_time_i.split(' ')[0]
    timerange = date_time_i.replace(date_time_f, '').strip()
    starttime, endtime = timerange.split(' - ')

    start_utc = f"{date_time_f}T{starttime.strip()}:00"
    end_utc = f"{date_time_f}T{endtime.strip()}:00"

    #make astropy time objects
    t_start = Time(start_utc, format='isot', scale='utc')
    t_end = Time(end_utc, format='isot', scale='utc')

    #add the buffer
    t_start_buffered = t_start - buffer * u.second
    t_end_buffered = t_end + buffer * u.second

    #format for gregorian input to fermi site
    fermi_start = t_start_buffered.strftime('%Y-%m-%d %H:%M:%S')
    fermi_end = t_end_buffered.strftime('%Y-%m-%d %H:%M:%S')

    #get sun coords at start time
    sun_position = get_sun(t_start)
    ra = round(sun_position.ra.deg, 3)
    dec = round(sun_position.dec.deg, 3)

    #print the parameters for the query
    print(f" --- Flare Name: {flarename} --- ")
    print(f"Object Coordinates      :{ra}, {dec}")
    print(f"Coordinate System       :J2000")
    print(f"Search Radius           :20 degrees")
    print(f"Obs. Dates              :{fermi_start} to {fermi_end}")
    print(f"Time System             :Gregorian")
    print(f"Energy Range            :100, 100000")
    print(f"LAT Data Type            :Photon")
    print(f"Always Check Spacecraft Data.")

Fermi LAT Data Query Parameters:
 --- Flare Name: 110906925 --- 
Object Coordinates      :165.08, 6.369
Coordinate System       :J2000
Search Radius           :20 degrees
Obs. Dates              :2011-09-06 22:10:50 to 2011-09-06 22:47:10
Time System             :Gregorian
Energy Range            :100, 100000
LAT Data Type            :Photon
Always Check Spacecraft Data.
 --- Flare Name: 120307028 --- 
Object Coordinates      :347.738, -5.261
Coordinate System       :J2000
Search Radius           :20 degrees
Obs. Dates              :2012-03-07 00:39:50 to 2012-03-07 01:20:10
Time System             :Gregorian
Energy Range            :100, 100000
LAT Data Type            :Photon
Always Check Spacecraft Data.
 --- Flare Name: 120307161 --- 
Object Coordinates      :347.861, -5.209
Coordinate System       :J2000
Search Radius           :20 degrees
Obs. Dates              :2012-03-07 03:50:50 to 2012-03-07 04:31:10
Time System             :Gregorian
Energy Range            :100, 100000
LAT

Now, I want to start making a csv that puts together the fits files into a "raw" csv to start from. 

In [6]:
#set raw data directory and output csv path
raw_dir = Path("data/fits/raw")
out_csv = Path("data/fits/raw/rawdata.csv")

#begin input for loop to read the fits files get the right info 
rows = []
for fp in sorted(raw_dir.rglob("*")):
    if not fp.is_file():
        continue
    if not (fp.name.lower().endswith(".fits") or fp.name.lower().endswith("fits.gz")):
        continue

#setup recording
    record = {
        "filename": fp.name,
        "filepath": str(fp), 
        "filesize": fp.stat().st_size, 
        "date_obs": None, 
        "date_end": None, 
        "ra": None, 
        "dec": None, 
        "instrument": None,
        "telescope": None, 
    }
#i would include object and flare class but this is not included 

    #read the fits file and extract the header
    try:
        with fits.open(fp) as hdul:
            header = hdul[0].header
            record["date_obs"] = header.get("DATE-OBS")
            record["date_end"] = header.get("DATE-END")
            record["ra"] = header.get("RA")
            record["dec"] = header.get("DEC")
            record["instrument"] = header.get("INSTRUME")
            record["telescope"] = header.get("TELESCOP")
    except Exception as e:
        record["error"] = str(e)
    
    #append the record to the list
    rows.append(record)

#create a dataframe from the records and write to csv
df_raw = pd.DataFrame(rows)
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(out_csv, index=False)

#print the number of records written and show the first few rows of the dataframe
print(f"Wrote {len(df_raw)} records to {out_csv}")
df_raw.head()



Wrote 2 records to data/fits/raw/rawdata.csv


,filename,filepath,filesize,date_obs,date_end,ra,dec,instrument,telescope
0,L2603311255089B9590BF64_PH00.fits,data/fits/raw/L2603311255089B9590BF64_PH00.fits,528920,2014-09-01T10:50:00.0000,2014-09-01T11:30:00.0000,None,None,LAT,GLAST
1,L2603311255089B9590BF64_SC00.fits,data/fits/raw/L2603311255089B9590BF64_SC00.fits,37813,2014-09-01T10:50:00.0000,2014-09-01T11:30:00.0000,None,None,LAT,GLAST
